In [ ]:
# [셀 1] 패키지 설치 및 cloudflared 준비
!pip install -q faster-whisper flask flask-cors
# cloudflared 설치 (Linux용)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("[*] 패키지 설치 완료.")

In [ ]:
# [셀 2] faster-whisper 모델 로드
import torch
from faster_whisper import WhisperModel

# GPU 가속을 위한 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"[*] 모델 로드 중 (device={device}, compute_type={compute_type})... ")
model = WhisperModel("large-v3", device=device, compute_type=compute_type)
print("[*] faster-whisper 'large-v3' 모델 로드 완료.")

In [ ]:
# [셀 3] Flask 서버 정의 및 전사 로직 구현
import os
import time
import tempfile
import traceback
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        "status": "ok",
        "model": "large-v3",
        "engine": "faster-whisper"
    })

@app.route('/transcribe', methods=['POST'])
def transcribe():
    if 'file' not in request.files:
        return jsonify({"error": "No file part"}), 400
    
    audio_file = request.files['file']
    if audio_file.filename == '':
        return jsonify({"error": "No selected file"}), 400

    temp_path = None
    try:
        # 임시 파일 저장
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp:
            audio_file.save(tmp.name)
            temp_path = tmp.name
        file_size = os.path.getsize(temp_path)
        print(f"[*] 파일 수신: {audio_file.filename}, 크기: {file_size}bytes")

        print(f"[*] 전사 시작: {audio_file.filename}")
        
        # faster-whisper 전사 실행
        print("[*] faster-whisper 전사 중...")
        segments, info = model.transcribe(
            temp_path, 
            language="ko", 
            beam_size=5,
            word_timestamps=False
        )

        # 결과 수집
        results = []
        full_text = ""
        for segment in segments:
            results.append({
                "start": float(segment.start),
                "end": float(segment.end),
                "text": segment.text.strip()
            })
            full_text += segment.text + " "

        print(f"[*] 전사 완료. 감지된 언어: {info.language}")

        return jsonify({
            "text": full_text.strip(),
            "segments": results,
            "language": info.language
        })

    except Exception as e:
        traceback.print_exc()
        print(f"[!] 오류: {str(e)}")
        return jsonify({"error": str(e)}), 500
    
    finally:
        # 임시 파일 정리
        if temp_path and os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except:
                pass

print("[*] Flask 서버 정의 완료.")

In [ ]:
# [셀 4] cloudflared 터널 시작 및 Flask 서버 실행
import threading
import subprocess
import re
import time

def run_tunnel():
    # 5000 포트로 Quick Tunnel 시작
    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    # URL 찾기
    print("\n" + "="*60)
    print("[*] Cloudflare 터널 생성 중...")
    
    url_found = False
    for line in process.stdout:
        # trycloudflare.com URL 추출
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print("\n" + "!"*60)
            print(f"  복사하여 앱에 입력할 URL:")
            print(f"  {tunnel_url}")
            print("!"*60 + "\n")
            url_found = True
            break
    
    if not url_found:
        print("[!] 터널 URL을 찾지 못했습니다. 출력을 확인하세요.")

# 터널을 별도 스레드에서 실행
threading.Thread(target=run_tunnel, daemon=True).start()

# Flask 서버 실행 (메인 스레드 점유)
print("[*] Flask 서버 시작 중 (Port 5000)...")
app.run(port=5000, host='0.0.0.0', debug=False, use_reloader=False)